# SwitchCraft on Colab / Jupyter (H100)

Runs the PyCharm **"Switchcraft: 2MHQ Positive Allostery"** run config
(`switchcraft.py --config tasks/pos_allostery.yaml --verbose`, `PYTHONUNBUFFERED=1`)
on an **H100 (80 GB)** — full settings, no memory trimming needed.

> **On Colab:** `Runtime` → `Change runtime type` → select your **H100** (or A100) GPU runtime.
>
> 80 GB of VRAM comfortably runs every example task in `tasks/` at full recycles, including the
> heavier multi-state ones (`ligand_discrimination`, `motif_switching`).

**Not on Colab?** On any box with the H100 + CUDA you can skip the notebook entirely and run the
config directly (e.g. `uv run python switchcraft.py --config tasks/pos_allostery.yaml --verbose`).
The cells below are just the clone → install → download-weights → run sequence in notebook form.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}  |  VRAM: {p.total_memory/1e9:.1f} GB")
else:
    print("No GPU — set Runtime → Change runtime type → GPU, then rerun.")

## 2. Clone the repo

In [ ]:
![ -d /content/switchcraft ] || git clone https://github.com/bauerdrpi/switchcraft.git /content/switchcraft
%cd /content/switchcraft
!ls

## 3. Install dependencies

We use the host's **preinstalled CUDA torch** (Boltz only needs `torch>=2.2`), so no multi-GB torch reinstall.

Two things to know:
- This Boltz fork imports `cuequivariance_torch` but doesn't declare it as a dependency, so we add it here.
- Boltz pins specific `numpy`/`scipy` versions, so pip may ask to **restart the runtime**. If it does:
  `Runtime` → `Restart session`, then **re-run this cell** and continue (the clone persists).

In [ ]:
%cd /content/switchcraft
# Boltz + its deps (keeps the host's CUDA torch)
!pip install -q -e boltz/
# SwitchCraft's prody + the missing cuequivariance-torch import
!pip install -q prody cuequivariance-torch
# Optional: pin the README's exact torch build only if you hit CUDA-op errors (~2.5GB, may need a restart):
# !pip install "torch==2.7.1+cu126" --index-url https://download.pytorch.org/whl/cu126
print("install done — if pip printed a RESTART warning, restart the runtime and re-run this cell")

## 4. Download the Boltz weights (~3.7 GB)

Downloaded into `boltz/` (the cache dir). On Colab this storage is **ephemeral** — it is re-downloaded
each new session. Takes a few minutes. (Mount Google Drive and point the cache there if you want to keep it.)

In [ ]:
%cd /content/switchcraft
from pathlib import Path
from boltz.main import download
cache = Path("boltz")
cache.mkdir(parents=True, exist_ok=True)
download(cache)
import zipfile
print("checkpoint valid zip:", zipfile.is_zipfile("boltz/boltz1_conf.ckpt"))

## 5. Run the design — PyCharm "2MHQ Positive Allostery" config

Exactly the run config: `switchcraft.py --config tasks/pos_allostery.yaml --verbose` with `PYTHONUNBUFFERED=1`.

On an H100 (80 GB) memory is not a concern — run at full settings. To try other designs, swap `--config`
for any file in `tasks/` (e.g. `tasks/ligand_discrimination.yaml`), or add `--num_designs N`.

In [ ]:
%cd /content/switchcraft
!PYTHONUNBUFFERED=1 python switchcraft.py --config tasks/pos_allostery.yaml --verbose
# Other tasks (H100 has plenty of headroom), e.g.:
# !PYTHONUNBUFFERED=1 python switchcraft.py --config tasks/ligand_discrimination.yaml --verbose --num_designs 4

## 6. View / download the outputs

Designs are written as `state<i>_sample<j>.pdb` (+ `.cif`, `.pkl`).

In [ ]:
%cd /content/switchcraft
import glob, zipfile
outs = sorted(set(glob.glob("**/*.pdb", recursive=True)) | set(glob.glob("**/*.cif", recursive=True)))
print(f"{len(outs)} structure file(s):")
for f in outs[:30]:
    print(" ", f)
if outs:
    with zipfile.ZipFile("designs.zip", "w") as z:
        for f in outs:
            z.write(f)
    try:
        from google.colab import files
        files.download("designs.zip")
    except Exception as e:
        print("zipped to designs.zip (download manually):", e)